# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library and Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets and their IDs using the Croissant metadata. Each record set, field, and column is referenced by its `@id` for precise, reproducible analysis.

In [ ]:
# List available record sets and their @id values
record_sets = metadata.record_sets
print('Available Record Sets:')
for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', '-')}")
    print('  Fields:')
    for field in rs.fields:
        print(f"    - Field Name: {field.name} (@id: {field.id}, dataType: {field.data_type})")
        if hasattr(field, 'columns'):
            for col in field.columns:
                print(f"       > Column: {col.name} (@id: {col.id}, dataType: {col.data_type})")
    print()

# Display a sample of records from each record set
for rs in record_sets:
    print(f"\nFirst record in '{rs.name}' (record_set id: {rs.id}):")
    try:
        recs = dataset.records(record_set=rs.id)
        print(next(recs))
    except StopIteration:
        print('  [No records found.]')
    except Exception as ex:
        print(f'  [Error loading records]: {ex}')

## 3. Data Extraction
Load data from each record set into a Pandas DataFrame for further analysis, referencing them by their `@id`.

In [ ]:
# Prepare dataframes for each record set using their @id
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set @id: {record_set_id} with shape {dataframes[record_set_id].shape}")
    except Exception as e:
        print(f"Could not load records for record set @id '{record_set_id}': {e}")

# Preview the columns and head of the first record set
if len(record_set_ids) > 0 and record_set_ids[0] in dataframes:
    first_rs_id = record_set_ids[0]
    print(f"\nColumns for record set @id {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print('No record sets found or loaded.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping using field `@id`s. Refer to the schema overview for suitable numeric and group fields.

In [ ]:
# Example: Filter and normalize a numeric field in the first available record set
import numpy as np

if len(dataframes) > 0:
    # Use the first DataFrame for demonstration
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]
    
    # Try to find a numeric field using Croissant field definitions
    rs_meta = next((rs for rs in metadata.record_sets if rs.id == record_set_id), None)
    numeric_fields = [f for f in rs_meta.fields if getattr(f, 'data_type', None) in ('Float', 'Integer', 'Number')]
    group_fields = [f for f in rs_meta.fields if getattr(f, 'data_type', None) in ('Text', 'String', 'Category')]
    
    if len(numeric_fields) == 0:
        print('No numeric fields found for EDA!')
    else:
        numeric_field_id = numeric_fields[0].id
        print(f'Using numeric field: {numeric_fields[0].name} (@id: {numeric_field_id})')

        # Handle possible missing/NaN values
        df_num = df.dropna(subset=[numeric_field_id]).copy()
        # Example threshold for filtering
        threshold = df_num[numeric_field_id].mean() if np.issubdtype(df_num[numeric_field_id].dtype, np.number) else 0
        try:
            filtered_df = df_num[df_num[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
            print(filtered_df.head())

            # Normalization
            normalized_col = f"{numeric_field_id}_normalized"
            filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"\nNormalized '{numeric_field_id}' for filtered records:")
            print(filtered_df[[numeric_field_id, normalized_col]].head())

            # Optional grouping
            if len(group_fields) > 0:
                group_field_id = group_fields[0].id
                print(f"\nGrouping by: {group_fields[0].name} (@id: {group_field_id})")
                if group_field_id in filtered_df.columns:
                    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
                    print(grouped.head())
                else:
                    print(f"Group field {group_field_id} not present in DataFrame columns.")
        except Exception as ex:
            print(f"EDA failed: {ex}")
else:
    print('No dataframes available to analyze.')

## 5. Visualization
Visualize field distributions or relationships. Update `numeric_field_id` and `group_field_id` as appropriate for your analysis.

In [ ]:
# Example: Visualize the distribution of the numeric field across groups
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and 'filtered_df' in locals():
    try:
        fig, ax = plt.subplots(figsize=(6, 4))
        if 'group_field_id' in locals() and group_field_id in filtered_df.columns:
            sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id], ax=ax)
            ax.set_xlabel(group_field_id)
        else:
            sns.histplot(filtered_df[numeric_field_id], kde=True, ax=ax)
        ax.set_ylabel(numeric_field_id)
        ax.set_title(f'Distribution of {numeric_field_id}')
        plt.tight_layout()
        plt.show()
    except Exception as ex:
        print(f"Visualization failed: {ex}")
else:
    print('No data available for visualization.')

## 6. Conclusion
This notebook demonstrated how to load, process, and visualize the FAIR² dataset using the Croissant schema and `mlcroissant`.

- We loaded metadata and records using the official schema (`@id` referencing ensures transparency and reproducibility).
- Explored available record sets, fields, and their types.
- Performed EDA by filtering, normalizing, and grouping numeric fields.
- Visualized key metrics to reveal patterns in rangeland adoption predictors.

**Next steps:** Adapt the EDA and visualization logic for domain-specific insights; consult metadata and Croissant schema for rich field documentation.